# Data Preparation — Satellite Image Classifier

Tile extraction, band selection (RGB+NIR), normalization,
and data augmentation pipeline for CNN training.

In [ ]:
import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path
import matplotlib.pyplot as plt

## Data Preparation Steps

1. Extract RGB + NIR bands from GeoTIFF tiles
2. Normalize pixel values using per-band statistics
3. Define augmentation pipeline (flips, rotations, color jitter)
4. Create train/val/test splits (70/15/15)
5. Build PyTorch Dataset and DataLoader

In [ ]:
# Custom dataset for satellite tiles
class SatelliteDataset(Dataset):
    def __init__(self, tile_paths, labels, transform=None):
        self.tile_paths = tile_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.tile_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.tile_paths[idx]) as src:
            # Select RGB + NIR (bands 1-4)
            img = src.read([1, 2, 3, 4]).astype(np.float32)
        # Normalize to [0, 1]
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = torch.from_numpy(img)
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

print('SatelliteDataset class defined')

In [ ]:
# Augmentation pipeline
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.Resize((224, 224)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
])

# Build datasets (placeholder paths)
# train_ds = SatelliteDataset(train_paths, train_labels, train_transform)
# val_ds = SatelliteDataset(val_paths, val_labels, val_transform)
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
print('Augmentation pipeline configured')